# Laparoscopic Surgery JSON Log Preprocessing for Advanced RAG

This notebook prepares laparoscopic surgery event-log JSON files for a production-style RAG pipeline.

It performs:

1. JSON folder ingestion  
2. Schema validation  
3. Event normalization  
4. Case-level metadata extraction  
5. Instrument, pedal, timing, and patient information extraction  
6. Case-level chunk creation  
7. Event-level chunk creation  
8. Final `chunks + metadata` export for embedding generation and FAISS indexing  

The output of this notebook is a clean dataset that can be used in the next step:

```text
chunks + metadata → embeddings → FAISS/vector database → hybrid retrieval → LLM grounded answer generation
```

## What schema validation means here

Your JSON files are event logs. Each file should contain a list of events like this:

```json
{
  "time": "2026-05-21T09:31:46",
  "event": "Surgery type selected",
  "value": "Cholecystectomy"
}
```

This notebook checks:

### File-level validation
- The file is valid JSON.
- The root object is a list.
- The file contains at least one event.
- The case has key events such as surgery type, surgeon name, and surgery start time.

### Event-level validation
- Every event is a dictionary.
- Every event has `time`, `event`, and `value` keys.
- `time` can be parsed as datetime.
- `event` is not empty.
- `value` is allowed to be empty for events such as instrument removed.

### Data-quality warnings
These do not stop the pipeline, but they are stored in a validation report:

- Missing patient info
- Missing surgery stopped event
- Missing explicit surgery duration
- Out-of-order timestamps
- Missing instrument count or connected duration
- Duplicate or repeated instrument summary events

For production, invalid files should be rejected or sent for manual review. For this prototype, valid records are processed and warnings are saved.

In [1]:
from pathlib import Path
import json
import re
from datetime import datetime
import pandas as pd
import numpy as np

## 1. Configuration

Put all your surgery JSON files in one folder and set `DATA_DIR`.

Example:

```text
data/surgery_json_logs/
    Dr.MERAI_Cholecystectomy_20260521_105445.json
    Cholecystectomy_2025-10-14_17-07-00.json
```

In [ ]:
# Change this path to your JSON folder.
# If you are running inside ChatGPT, the uploaded files may already be in /mnt/data.
DATA_DIR = Path("/home/corpadm/Desktop/Autonomous AI Robotics/RAG-workflow/Logs")

OUTPUT_DIR = Path("/home/corpadm/Desktop/Autonomous AI Robotics/RAG-workflow/processed_surgery_rag_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JSON_PATTERN = "*.json"

json_files = sorted(DATA_DIR.glob(JSON_PATTERN))
print(f"Found {len(json_files)} JSON files")
for p in json_files[:5]:  # Print the first 5 JSON files
    print("-", p.name)

Found 16 JSON files
- Adrenalectomy_2025-10-14_16-12-00.json
- Cholecystectomy_2025-10-14_17-07-00.json
- Cholecystectomy_2025-10-16_16-59-00.json
- Dr.MERAI_Cholecystectomy_20260521_105445.json
- DrRaj_20260714_Cholecystectomy.json
- Gastric_Bypass_(Roux-en-Y_-_Mini)_2025-10-21_16-35-00.json
- Hiatal_Hernia_Repair_2025-10-13_18-41-00.json
- Hiatal_Hernia_Repair_2025-10-19_16-13-00.json
- Hysterectomy_2025-10-13_16-59-00.json
- Liver_Resection_(Hepatectomy)_2025-10-20_18-40-00.json


## 2. Helper functions

In [3]:
def normalize_event_name(event: str) -> str:
    # Normalize event name for easier matching.
    if event is None:
        return ""
    event = str(event).strip()
    event = re.sub(r"\s+", " ", event)
    return event


def parse_datetime(value):
    # Parse datetime from ISO or common timestamp string.
    if value is None or str(value).strip() == "":
        return pd.NaT

    value = str(value).strip()
    formats = [
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%dT%H:%M:%S.%f",
        "%Y-%m-%d %H:%M:%S.%f",
    ]

    for fmt in formats:
        try:
            return datetime.strptime(value, fmt)
        except ValueError:
            pass

    try:
        return pd.to_datetime(value, errors="coerce")
    except Exception:
        return pd.NaT


def parse_patient_info(text):
    # Parse patient info such as: 'Name: XXXX, Age: 54, BMI: 62.5'
    result = {
        "patient_name": None,
        "patient_age": None,
        "patient_bmi": None
    }

    if not isinstance(text, str):
        return result

    name_match = re.search(r"Name\s*:\s*([^,]+)", text, flags=re.I)
    age_match = re.search(r"Age\s*:\s*([0-9]+)", text, flags=re.I)
    bmi_match = re.search(r"BMI\s*:\s*([0-9]+(?:\.[0-9]+)?)", text, flags=re.I)

    if name_match:
        result["patient_name"] = name_match.group(1).strip()
    if age_match:
        result["patient_age"] = int(age_match.group(1))
    if bmi_match:
        result["patient_bmi"] = float(bmi_match.group(1))

    return result


def safe_first(events, event_name):
    # Return first value for a given normalized event name.
    matches = events[events["event_norm"].str.lower() == event_name.lower()]
    if len(matches) == 0:
        return None
    return matches.iloc[0]["value"]


def safe_last(events, event_name):
    # Return last value for a given normalized event name.
    matches = events[events["event_norm"].str.lower() == event_name.lower()]
    if len(matches) == 0:
        return None
    return matches.iloc[-1]["value"]

## 3. Schema validation

The function below validates each JSON file and returns:

- `events_df`: cleaned event table
- `issues`: validation errors/warnings
- `is_valid`: whether the file can be processed

Hard errors stop a file from processing. Warnings are saved but the file can still be used.

In [4]:
REQUIRED_EVENT_KEYS = {"time", "event", "value"}

IMPORTANT_CASE_EVENTS = [
    "Surgery type selected",
    "Surgeon Name",
    "Surgery started"
]


def validate_json_file(file_path: Path):
    issues = []

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            raw = json.load(f)
    except Exception as e:
        return None, [{"severity": "error", "issue": f"Invalid JSON: {e}"}], False

    if not isinstance(raw, list):
        return None, [{"severity": "error", "issue": "Root JSON object must be a list of events"}], False

    if len(raw) == 0:
        return None, [{"severity": "error", "issue": "JSON event list is empty"}], False

    cleaned_rows = []

    for i, item in enumerate(raw):
        if not isinstance(item, dict):
            issues.append({
                "severity": "error",
                "row_index": i,
                "issue": "Event must be a dictionary/object"
            })
            continue

        missing_keys = REQUIRED_EVENT_KEYS - set(item.keys())
        if missing_keys:
            issues.append({
                "severity": "error",
                "row_index": i,
                "issue": f"Missing required keys: {sorted(missing_keys)}"
            })
            continue

        event_time = parse_datetime(item.get("time"))
        event_name = normalize_event_name(item.get("event"))
        value = item.get("value", "")

        if pd.isna(event_time):
            issues.append({
                "severity": "error",
                "row_index": i,
                "issue": f"Invalid timestamp: {item.get('time')}"
            })

        if event_name == "":
            issues.append({
                "severity": "error",
                "row_index": i,
                "issue": "Event name is empty"
            })

        cleaned_rows.append({
            "row_index": i,
            "time": event_time,
            "event": item.get("event"),
            "event_norm": event_name,
            "value": "" if value is None else str(value).strip(),
            "source_file": file_path.name
        })

    events_df = pd.DataFrame(cleaned_rows)
    has_error = any(x["severity"] == "error" for x in issues)

    if events_df.empty:
        issues.append({"severity": "error", "issue": "No usable event rows after cleaning"})
        return events_df, issues, False

    existing_events = set(events_df["event_norm"].str.lower().tolist())
    for ev in IMPORTANT_CASE_EVENTS:
        if ev.lower() not in existing_events:
            issues.append({
                "severity": "warning",
                "issue": f"Missing important case event: {ev}"
            })

    optional_events = ["Patient Info", "Surgery stopped", "Surgery duration"]
    for ev in optional_events:
        if ev.lower() not in existing_events:
            issues.append({
                "severity": "warning",
                "issue": f"Missing optional event: {ev}"
            })

    if not events_df["time"].is_monotonic_increasing:
        issues.append({
            "severity": "warning",
            "issue": "Timestamps are not strictly sorted; sorting will be applied for timeline features"
        })

    return events_df, issues, not has_error

## 4. Metadata extraction

This extracts one metadata row per surgery case.

Metadata is useful for hybrid retrieval because some questions are exact-filter questions:

```text
Show cases by Dr.MERAI
Find Cholecystectomy cases
Which case used Medium_Large_Clip_Applier?
Which case had BMI above 30?
```

In [5]:
def extract_instrument_summary(events_df):
    rows = []
    current = {}

    sorted_df = events_df.sort_values("time").reset_index(drop=True)

    for _, row in sorted_df.iterrows():
        event = row["event_norm"]
        value = row["value"]
        time = row["time"]

        side = None
        if "PrimaryLeft" in event:
            side = "PrimaryLeft"
        elif "PrimaryRight" in event:
            side = "PrimaryRight"

        if side is None:
            continue

        if "Instrument Name" in event:
            if current.get("instrument_name") and ("count" in current or "connected_duration_seconds" in current):
                rows.append(current)
            current = {
                "side": side,
                "instrument_name": value,
                "summary_time": time
            }

        elif "Instrument Count" in event:
            if not current:
                current = {"side": side, "summary_time": time}
            try:
                current["count"] = int(float(value))
            except Exception:
                current["count"] = None

        elif "Instrument Connected duration" in event:
            if not current:
                current = {"side": side, "summary_time": time}
            try:
                current["connected_duration_seconds"] = float(value)
            except Exception:
                current["connected_duration_seconds"] = None

            rows.append(current)
            current = {}

    if current:
        rows.append(current)

    clean_rows = []
    for r in rows:
        if r.get("instrument_name"):
            clean_rows.append(r)

    return clean_rows


def extract_case_metadata(events_df, file_path: Path):
    events_sorted = events_df.sort_values("time").reset_index(drop=True)

    surgery_type = safe_first(events_sorted, "Surgery type selected")
    surgeon_name = safe_first(events_sorted, "Surgeon Name")
    patient_info_raw = safe_first(events_sorted, "Patient Info")
    patient = parse_patient_info(patient_info_raw)

    surgery_started = safe_first(events_sorted, "Surgery started")
    surgery_stopped = safe_last(events_sorted, "Surgery stopped")
    surgery_duration = safe_last(events_sorted, "Surgery duration")

    start_dt = parse_datetime(surgery_started) if surgery_started else pd.NaT
    stop_dt = parse_datetime(surgery_stopped) if surgery_stopped else pd.NaT

    computed_duration_minutes = None
    if pd.notna(start_dt) and pd.notna(stop_dt):
        computed_duration_minutes = round((stop_dt - start_dt).total_seconds() / 60, 2)

    clutch_count = events_sorted["event_norm"].str.contains("Clutch Pedal Pressed", case=False, na=False).sum()
    camera_count = events_sorted["event_norm"].str.contains("Camera Pedal Pressed", case=False, na=False).sum()

    instrument_rows = extract_instrument_summary(events_sorted)
    instrument_names = sorted(set(
        r.get("instrument_name") for r in instrument_rows if r.get("instrument_name")
    ))

    live_instruments = events_sorted[
        events_sorted["event_norm"].str.contains("Instrument Name", case=False, na=False)
    ]["value"].dropna().astype(str).str.strip().tolist()

    all_instruments = sorted(set(instrument_names + live_instruments))

    case_id = file_path.stem

    meta = {
        "case_id": case_id,
        "source_file": file_path.name,
        "surgery_type": surgery_type,
        "surgeon_name": surgeon_name.strip() if isinstance(surgeon_name, str) else surgeon_name,
        "patient_info_raw": patient_info_raw,
        "patient_name": patient["patient_name"],
        "patient_age": patient["patient_age"],
        "patient_bmi": patient["patient_bmi"],
        "surgery_started": surgery_started,
        "surgery_stopped": surgery_stopped,
        "surgery_duration_raw": surgery_duration,
        "computed_duration_minutes": computed_duration_minutes,
        "first_event_time": events_sorted["time"].min(),
        "last_event_time": events_sorted["time"].max(),
        "total_events": len(events_sorted),
        "clutch_pedal_count": int(clutch_count),
        "camera_pedal_count": int(camera_count),
        "instrument_names": all_instruments,
        "instrument_count_unique": len(all_instruments),
        "instrument_summary_rows": instrument_rows,
    }

    return meta

## 5. Chunk creation

This notebook creates two chunk levels.

### A. Case-level chunk
One chunk per surgery file. Best for broad questions:

```text
Summarize this case.
Who was the surgeon?
What instruments were used?
How many pedal events occurred?
```

### B. Event-level chunks
Multiple chunks per surgery file. Best for detailed questions:

```text
Which instrument had the longest connected duration?
How many camera pedal events happened?
When was the surgery started?
What patient BMI was recorded?
```

In [6]:
def make_case_level_chunk(meta):
    instruments = ", ".join(meta["instrument_names"]) if meta["instrument_names"] else "not recorded"

    duration_text = meta["surgery_duration_raw"]
    if not duration_text and meta["computed_duration_minutes"] is not None:
        duration_text = f"{meta['computed_duration_minutes']} minutes"
    if not duration_text:
        duration_text = "not recorded"

    text = (
        f"Case {meta['case_id']} is a laparoscopic surgery log. "
        f"Surgery type: {meta.get('surgery_type') or 'not recorded'}. "
        f"Surgeon: {meta.get('surgeon_name') or 'not recorded'}. "
        f"Patient age: {meta.get('patient_age') or 'not recorded'}, "
        f"BMI: {meta.get('patient_bmi') or 'not recorded'}. "
        f"Surgery started: {meta.get('surgery_started') or 'not recorded'}. "
        f"Surgery stopped: {meta.get('surgery_stopped') or 'not recorded'}. "
        f"Surgery duration: {duration_text}. "
        f"Total events: {meta['total_events']}. "
        f"Clutch pedal presses: {meta['clutch_pedal_count']}. "
        f"Camera pedal presses: {meta['camera_pedal_count']}. "
        f"Instruments used: {instruments}."
    )

    return {
        "chunk_id": f"{meta['case_id']}__case_summary",
        "chunk_type": "case_summary",
        "case_id": meta["case_id"],
        "source_file": meta["source_file"],
        "surgery_type": meta["surgery_type"],
        "surgeon_name": meta["surgeon_name"],
        "surgery_date": str(meta["first_event_time"].date()) if pd.notna(meta["first_event_time"]) else None,
        "event_start_time": meta["first_event_time"],
        "event_end_time": meta["last_event_time"],
        "patient_age": meta["patient_age"],
        "patient_bmi": meta["patient_bmi"],
        "instrument_names": meta["instrument_names"],
        "text": text
    }


def make_patient_timing_chunk(meta):
    duration_text = meta["surgery_duration_raw"]
    if not duration_text and meta["computed_duration_minutes"] is not None:
        duration_text = f"{meta['computed_duration_minutes']} minutes"
    if not duration_text:
        duration_text = "not recorded"

    text = (
        f"Timing and patient details for case {meta['case_id']}. "
        f"Patient information: {meta.get('patient_info_raw') or 'not recorded'}. "
        f"Patient age: {meta.get('patient_age') or 'not recorded'}. "
        f"Patient BMI: {meta.get('patient_bmi') or 'not recorded'}. "
        f"Surgery started at {meta.get('surgery_started') or 'not recorded'} and "
        f"stopped at {meta.get('surgery_stopped') or 'not recorded'}. "
        f"Recorded surgery duration: {duration_text}."
    )

    return {
        "chunk_id": f"{meta['case_id']}__patient_timing",
        "chunk_type": "patient_timing",
        "case_id": meta["case_id"],
        "source_file": meta["source_file"],
        "surgery_type": meta["surgery_type"],
        "surgeon_name": meta["surgeon_name"],
        "surgery_date": str(meta["first_event_time"].date()) if pd.notna(meta["first_event_time"]) else None,
        "event_start_time": meta["first_event_time"],
        "event_end_time": meta["last_event_time"],
        "patient_age": meta["patient_age"],
        "patient_bmi": meta["patient_bmi"],
        "instrument_names": [],
        "text": text
    }


def make_pedal_activity_chunk(meta):
    text = (
        f"Pedal activity summary for case {meta['case_id']}. "
        f"The log contains {meta['clutch_pedal_count']} clutch pedal press events and "
        f"{meta['camera_pedal_count']} camera pedal press events. "
        f"These events describe intraoperative workflow activity during the surgery."
    )

    return {
        "chunk_id": f"{meta['case_id']}__pedal_activity",
        "chunk_type": "pedal_activity",
        "case_id": meta["case_id"],
        "source_file": meta["source_file"],
        "surgery_type": meta["surgery_type"],
        "surgeon_name": meta["surgeon_name"],
        "surgery_date": str(meta["first_event_time"].date()) if pd.notna(meta["first_event_time"]) else None,
        "event_start_time": meta["first_event_time"],
        "event_end_time": meta["last_event_time"],
        "patient_age": meta["patient_age"],
        "patient_bmi": meta["patient_bmi"],
        "instrument_names": [],
        "text": text
    }


def make_instrument_summary_chunks(meta):
    chunks = []
    rows = meta["instrument_summary_rows"]

    if not rows:
        instruments = ", ".join(meta["instrument_names"]) if meta["instrument_names"] else "not recorded"
        text = (
            f"Instrument summary for case {meta['case_id']}. "
            f"Instruments recorded in the log: {instruments}. "
            f"Detailed connected duration and count values were not fully available."
        )
        chunks.append({
            "chunk_id": f"{meta['case_id']}__instrument_summary",
            "chunk_type": "instrument_summary",
            "case_id": meta["case_id"],
            "source_file": meta["source_file"],
            "surgery_type": meta["surgery_type"],
            "surgeon_name": meta["surgeon_name"],
            "surgery_date": str(meta["first_event_time"].date()) if pd.notna(meta["first_event_time"]) else None,
            "event_start_time": meta["first_event_time"],
            "event_end_time": meta["last_event_time"],
            "patient_age": meta["patient_age"],
            "patient_bmi": meta["patient_bmi"],
            "instrument_names": meta["instrument_names"],
            "text": text
        })
        return chunks

    summary_lines = []
    for r in rows:
        summary_lines.append(
            f"{r.get('instrument_name')} on {r.get('side', 'unknown side')} "
            f"had count {r.get('count', 'not recorded')} and connected duration "
            f"{r.get('connected_duration_seconds', 'not recorded')} seconds"
        )

    text = (
        f"Instrument usage summary for case {meta['case_id']}. "
        + ". ".join(summary_lines)
        + "."
    )

    chunks.append({
        "chunk_id": f"{meta['case_id']}__instrument_summary",
        "chunk_type": "instrument_summary",
        "case_id": meta["case_id"],
        "source_file": meta["source_file"],
        "surgery_type": meta["surgery_type"],
        "surgeon_name": meta["surgeon_name"],
        "surgery_date": str(meta["first_event_time"].date()) if pd.notna(meta["first_event_time"]) else None,
        "event_start_time": meta["first_event_time"],
        "event_end_time": meta["last_event_time"],
        "patient_age": meta["patient_age"],
        "patient_bmi": meta["patient_bmi"],
        "instrument_names": meta["instrument_names"],
        "text": text
    })

    for i, r in enumerate(rows):
        inst = r.get("instrument_name")
        text = (
            f"Instrument event for case {meta['case_id']}. "
            f"Instrument name: {inst}. "
            f"Side: {r.get('side', 'not recorded')}. "
            f"Count: {r.get('count', 'not recorded')}. "
            f"Connected duration: {r.get('connected_duration_seconds', 'not recorded')} seconds. "
            f"Summary time: {r.get('summary_time', 'not recorded')}."
        )
        chunks.append({
            "chunk_id": f"{meta['case_id']}__instrument_{i+1}",
            "chunk_type": "instrument_event",
            "case_id": meta["case_id"],
            "source_file": meta["source_file"],
            "surgery_type": meta["surgery_type"],
            "surgeon_name": meta["surgeon_name"],
            "surgery_date": str(meta["first_event_time"].date()) if pd.notna(meta["first_event_time"]) else None,
            "event_start_time": r.get("summary_time"),
            "event_end_time": r.get("summary_time"),
            "patient_age": meta["patient_age"],
            "patient_bmi": meta["patient_bmi"],
            "instrument_names": [inst] if inst else [],
            "text": text
        })

    return chunks


def create_chunks_for_case(meta):
    chunks = []
    chunks.append(make_case_level_chunk(meta))
    chunks.append(make_patient_timing_chunk(meta))
    chunks.append(make_pedal_activity_chunk(meta))
    chunks.extend(make_instrument_summary_chunks(meta))
    return chunks

## 6. Run preprocessing on all JSON files

In [7]:
all_metadata = []
all_chunks = []
validation_rows = []

for file_path in json_files:
    events_df, issues, is_valid = validate_json_file(file_path)

    for issue in issues:
        validation_rows.append({
            "source_file": file_path.name,
            "severity": issue.get("severity"),
            "row_index": issue.get("row_index"),
            "issue": issue.get("issue")
        })

    if not is_valid:
        print(f"Skipping invalid file: {file_path.name}")
        continue

    meta = extract_case_metadata(events_df, file_path)
    chunks = create_chunks_for_case(meta)

    meta_export = meta.copy()
    meta_export["instrument_names"] = json.dumps(meta_export["instrument_names"])
    meta_export["instrument_summary_rows"] = json.dumps(meta_export["instrument_summary_rows"], default=str)
    all_metadata.append(meta_export)

    all_chunks.extend(chunks)

metadata_df = pd.DataFrame(all_metadata)
chunks_df = pd.DataFrame(all_chunks)
validation_df = pd.DataFrame(validation_rows)

print("Processed cases:", len(metadata_df))
print("Created chunks:", len(chunks_df))
print("Validation issues:", len(validation_df))

display(metadata_df.head())
display(chunks_df[["chunk_id", "chunk_type", "case_id", "surgery_type", "surgeon_name", "text"]].head(10))

Processed cases: 16
Created chunks: 152
Validation issues: 17


,case_id,source_file,surgery_type,surgeon_name,patient_info_raw,patient_name,patient_age,patient_bmi,surgery_started,surgery_stopped,surgery_duration_raw,computed_duration_minutes,first_event_time,last_event_time,total_events,clutch_pedal_count,camera_pedal_count,instrument_names,instrument_count_unique,instrument_summary_rows
0,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy_2025-10-14_16-12-00.json,Adrenalectomy,Dr.Meril M,"Name: Patient_2, Age: 59, BMI: 20.0",Patient_2,59,20.0,2025-10-14 16:12:00,2025-10-14 16:55:00,0:43:00,43.0,2025-10-14 16:12:00,2025-10-14 16:55:00,87,60,0,"[""Cobra_Grasper"", ""Fenestrated_Bipolar_Forceps...",6,"[{""side"": ""PrimaryLeft"", ""instrument_name"": ""M..."
1,Cholecystectomy_2025-10-14_17-07-00,Cholecystectomy_2025-10-14_17-07-00.json,Cholecystectomy,Dr.Meril M,"Name: Patient_2, Age: 50, BMI: 28.9",Patient_2,50,28.9,2025-10-14 17:07:00,2025-10-14 17:43:00,0:36:00,36.0,2025-10-14 17:07:00,2025-10-14 17:43:00,80,53,0,"[""Large_Clip_Applier"", ""Long_Bipolar_Grasper"",...",6,"[{""side"": ""PrimaryLeft"", ""instrument_name"": ""L..."
2,Cholecystectomy_2025-10-16_16-59-00,Cholecystectomy_2025-10-16_16-59-00.json,Cholecystectomy,Dr.Meril M,"Name: Patient_4, Age: 33, BMI: 22.6",Patient_4,33,22.6,2025-10-16 16:59:00,2025-10-16 17:55:00,0:56:00,56.0,2025-10-16 16:59:00,2025-10-16 17:55:00,96,60,0,"[""Endowrist_Stapler_30_Curved_Tip_Instrument"",...",7,"[{""side"": ""PrimaryLeft"", ""instrument_name"": ""S..."
3,Dr.MERAI_Cholecystectomy_20260521_105445,Dr.MERAI_Cholecystectomy_20260521_105445.json,Cholecystectomy,Dr.MERAI,"Name: XXXX, Age: 54, BMI: 62.5",XXXX,54,62.5,2026-05-21 09:36:26,None,None,NaN,2026-05-21 09:31:46,2026-05-21 10:54:45,293,176,81,"[""Fenestrated_Bipolar_Forceps"", ""Medium_Large_...",3,"[{""side"": ""PrimaryRight"", ""instrument_name"": ""..."
4,DrRaj_20260714_Cholecystectomy,DrRaj_20260714_Cholecystectomy.json,Appendectomy,Dr.Meril M,"Name: Patient_4, Age: 58, BMI: 28.1",Patient_4,58,28.1,2025-10-16 17:54:00,2025-10-16 18:41:00,0:47:00,47.0,2025-10-16 17:54:00,2025-10-16 18:41:00,64,40,0,"[""Atrial_Retractor_Short_Right"", ""Black_Diamon...",6,"[{""side"": ""PrimaryLeft"", ""instrument_name"": ""P..."


,chunk_id,chunk_type,case_id,surgery_type,surgeon_name,text
0,Adrenalectomy_2025-10-14_16-12-00__case_summary,case_summary,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy,Dr.Meril M,Case Adrenalectomy_2025-10-14_16-12-00 is a la...
1,Adrenalectomy_2025-10-14_16-12-00__patient_timing,patient_timing,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy,Dr.Meril M,Timing and patient details for case Adrenalect...
2,Adrenalectomy_2025-10-14_16-12-00__pedal_activity,pedal_activity,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy,Dr.Meril M,Pedal activity summary for case Adrenalectomy_...
3,Adrenalectomy_2025-10-14_16-12-00__instrument_...,instrument_summary,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy,Dr.Meril M,Instrument usage summary for case Adrenalectom...
4,Adrenalectomy_2025-10-14_16-12-00__instrument_1,instrument_event,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy,Dr.Meril M,Instrument event for case Adrenalectomy_2025-1...
5,Adrenalectomy_2025-10-14_16-12-00__instrument_2,instrument_event,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy,Dr.Meril M,Instrument event for case Adrenalectomy_2025-1...
6,Adrenalectomy_2025-10-14_16-12-00__instrument_3,instrument_event,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy,Dr.Meril M,Instrument event for case Adrenalectomy_2025-1...
7,Adrenalectomy_2025-10-14_16-12-00__instrument_4,instrument_event,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy,Dr.Meril M,Instrument event for case Adrenalectomy_2025-1...
8,Adrenalectomy_2025-10-14_16-12-00__instrument_5,instrument_event,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy,Dr.Meril M,Instrument event for case Adrenalectomy_2025-1...
9,Adrenalectomy_2025-10-14_16-12-00__instrument_6,instrument_event,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy,Dr.Meril M,Instrument event for case Adrenalectomy_2025-1...


## 7. Save outputs

These files are created:

- `case_metadata.csv`: one row per surgery case
- `rag_chunks_metadata.csv`: chunks and metadata for embedding generation
- `rag_chunks_metadata.jsonl`: same content in JSONL format
- `validation_report.csv`: schema validation warnings/errors

Use `rag_chunks_metadata.csv` or `rag_chunks_metadata.jsonl` as the input for embedding generation.

In [8]:
chunks_export = chunks_df.copy()

for col in ["instrument_names"]:
    if col in chunks_export.columns:
        chunks_export[col] = chunks_export[col].apply(lambda x: json.dumps(x) if isinstance(x, list) else x)

for col in ["event_start_time", "event_end_time"]:
    if col in chunks_export.columns:
        chunks_export[col] = chunks_export[col].astype(str)

metadata_path = OUTPUT_DIR / "case_metadata.csv"
chunks_csv_path = OUTPUT_DIR / "rag_chunks_metadata.csv"
chunks_jsonl_path = OUTPUT_DIR / "rag_chunks_metadata.jsonl"
validation_path = OUTPUT_DIR / "validation_report.csv"

metadata_df.to_csv(metadata_path, index=False)
chunks_export.to_csv(chunks_csv_path, index=False)
validation_df.to_csv(validation_path, index=False)

with open(chunks_jsonl_path, "w", encoding="utf-8") as f:
    for _, row in chunks_export.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

print("Saved:")
print(metadata_path)
print(chunks_csv_path)
print(chunks_jsonl_path)
print(validation_path)

Saved:
/home/corpadm/Desktop/Autonomous AI Robotics/RAG-workflow/processed_surgery_rag_data/case_metadata.csv
/home/corpadm/Desktop/Autonomous AI Robotics/RAG-workflow/processed_surgery_rag_data/rag_chunks_metadata.csv
/home/corpadm/Desktop/Autonomous AI Robotics/RAG-workflow/processed_surgery_rag_data/rag_chunks_metadata.jsonl
/home/corpadm/Desktop/Autonomous AI Robotics/RAG-workflow/processed_surgery_rag_data/validation_report.csv


## 8. Quality checks before embedding generation

In [ ]:
print("Chunk type distribution:")
display(chunks_df["chunk_type"].value_counts())

print("\nMissing values in important metadata:")
important_cols = ["case_id", "surgery_type", "surgeon_name", "surgery_date", "text"]
display(chunks_df[important_cols].isna().sum())

print("\nSample chunk text:")
for i, row in chunks_df.head(5).iterrows():
    print("=" * 80)
    print("chunk_id:", row["chunk_id"])
    print("chunk_type:", row["chunk_type"])
    print(row["text"])